# 2.2 Compact Attention


In [1]:
import torch
import torch.nn as nn

## MHA
Multi-head attention. The original version of transformer paper.

$$ Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d}})V $$

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, num_head):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_head = num_head
        self.head_dim = hidden_size // num_head

        self.W_q = nn.Linear(hidden_size, hidden_size)
        self.W_k = nn.Linear(hidden_size, hidden_size)
        self.W_v = nn.Linear(hidden_size, hidden_size)

        self.out_proj = nn.Linear(hidden_size, hidden_size)

    def forward(self, X:torch.Tensor, mask=None):
        # input X: batch_size * seq_len * hidden_size
        batch_size, seq_len, _ = X.shape()

        # transform size into (batch_size, num_head, seq_len, head_dim)
        query = self.W_q(X).view(batch_size, seq_len, self.num_head, self.head_dim).transpose(1,2)
        key = self.W_k(X).view(batch_size, seq_len, self.num_head, self.head_dim).transpose(1,2)
        # todo: rope
        value = self.W_v(X).view(batch_size, seq_len, self.num_head, self.head_dim).transpose(1,2)

        # calculate attention score for every head
        p = torch.matmul(query, key.transpose(-2, -1))  / (self.head_dim ** 0.5)

        if mask is not None:
            p = p.masked_fill(mask, self.float('-inf'))

        p = torch.softmax(p, dim=-1) # batch_size, num_head, seq_len, seq_len
        o = torch.matmul(p, value) # batch_size, num_head, seq_len, head_dim

        # concat
        o = o.transpose(1,2).reshape(batch_size, seq_len, self.hidden_size)

        return self.out_proj(o)
        

## MQA

Multi-Query attention. Several query head with only **one** KV head.

In [ ]:
class MultiQueryAttention(nn.Module):
    def __init__(self, hidden_size, num_head):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_head = num_head
        self.head_dim = hidden_size // num_head

        self.W_q = nn.Linear(hidden_size, hidden_size)
        self.W_k = nn.Linear(hidden_size, self.head_dim)
        self.W_v = nn.Linear(hidden_size, self.head_dim)

        self.out_proj = nn.Linear(hidden_size, hidden_size)

    def forward(self, X: torch.Tensor, mask=None):
        batch_size, seq_len, _ = X.shape()

        query = self.W_q(X).view(batch_size, seq_len, self.num_head, self.head_dim).transpose(1,2) # batch_size, num_head, seq_len, head_dim
        key = self.W_k(X).unsqueeze(1).expand(-1, self.num_head, -1, -1)
        value = self.W_v(X).unsqueeze(1).expand(-1, self.num_head, -1, -1)

        p = torch.matmul(query, key.transpose(-2, -1)) / (self.head_dim ** 0.5)
        if mask is not None:
            p = p.masked_fill(mask, self.float('-inf'))
        p = torch.softmax(p, dim=-1)
        o = torch.matmul(p, value)

        # concat
        o = o.transpose(1,2).reshape(batch_size, seq_len, self.hidden_size)
        return self.out_proj(o)

## GQA
Group Query Attetion. Group several query heads with one KV head, but can have multiple KV head in LLM.

In [ ]:
class GroupQueryAttention(nn.Module):
    def __init__(self, hidden_size, num_head, group_size):
        # group_size: the number of query head in one group(with one KV head)
        super().__init__()

        assert hidden_size % num_head == 0, "hidden_size must be dividable with num_head"
        assert num_head % group_size == 0, "num_head must be dividable with group_size"

        self.hidden_size = hidden_size
        self.num_head = num_head
        self.head_dim = hidden_size // num_head
        self.num_kv_head = num_head // group_size
        self.group_size = group_size

        self.W_q = nn.Linear(hidden_size, hidden_size)
        self.W_k = nn.Linear(hidden_size, self.num_kv_head * self.head_dim)
        self.W_v = nn.Linear(hidden_size, self.num_kv_head * self.head_dim)

        self.out_proj = nn.Linear(hidden_size, hidden_size)

    def forward(self, X: torch.Tensor, mask=None):
        batch_size, seq_len, _ = X.shape()

        query = self.W_q(X).view(batch_size, seq_len, self.num_head, self.head_dim).transpose(1, 2)
        key = self.W_k(X).view(batch_size, seq_len, self.num_kv_head, self.head_dim).transpose(1, 2)
        key = key.unsqueeze(2).expand(-1,-1, self.group_size, -1, -1).reshape(batch_size, self.num_head, seq_len, self.head_dim)
        value = self.W_v(X).view(batch_size, seq_len, self.num_kv_head, self.head_dim).transpose(1, 2)
        value = value.unsqueeze(2).expand(-1,-1, self.group_size, -1, -1).reshape(batch_size, self.num_head, seq_len, self.head_dim)

        p = torch.matmul(query, key.transpose(-2, -1)) / (self.head_dim ** 0.5)
        if mask is not None:
            p = p.masked_fill(mask, self.float('-inf'))
        p = torch.softmax(p, dim=-1)
        o = torch.matmul(p, value)

        # concat
        o = o.transpose(1,2).reshape(batch_size, seq_len, self.hidden_size)
        return self.out_proj(o)


## MLA
![Compact Attentions](../figs/2.2-MLA.png)
*This picture is the comparison of different compact attention mechanisms. (From DeepSeek-v2 paper, see **Reference 2**)*

In Multi-head Latent Attention, KV cache is compressed via low-rank projection during storage, and up project into the original rank/dimention in actual attention caculation.

Let $Wq$ be the weight of query, $W_{Uk}$ be the up projection weight of key, $c_i$ is the compressed keys and $x_t$ is the new token input.

Then $q*k^T$ calculation in attention can be transformed into:
$$ q_tk_i^T = (x_t W_q)(c_i W_{uk})^T = x_t (W_q W_{Uk}^T) c_i^T $$

And the weight part can be pre-calculated in order to save computation time.

### RoPE in MLA

> For more RoPE, refer to *3.1-Position-Embedding*

RoPE enables the model to get better understanding of relative positions between tokens and thus leads to better performance. Simply applying rope in MLA will look like this:

$$ q_tk_i^T = (x_t W_q R_t)(c_i W_{uk} R_i)^T = x_t (W_q R_{t-i} W_{Uk}^T) c_i^T  $$

$R_{t-i}$ cannot be pre-calculated, and will drag the speed.

**Solution: (Decoupled RoPE) Using additional $q^R, k^R$ in dimention $d_h^R$ to keep RoPE and optimize calculation as well.**

$$ q_tk_i^T = [q^C; q^R][k^C; k^R]^T = q^C(k^C)^T + q^R(k^R)^T $$

$q^C(k^C)^T$ is the normal down-projection + up-projection. $q^R(k^R)^T$ is calculated like MQA by sharing the same $k^R$. And we get $q^R, k^R$ in the following way:
- $q^R = RoPE(C^QW^{QR})$, where $C^Q = XW^{DQ}$
- $k^R = RoPE(XW^{KR})$

> TODO: Why not using $C^K=XW^{DK}$ for key RoPE as well?

In [ ]:
class RotaryEmbedding(nn.Module):
    def __init__(self, hidden_size, num_heads, theta=10000, max_len=512):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.theta = theta
        self.max_len = max_len

        theta_i = 1.0 / self.theta ** (torch.arange(0, self.head_dim, 2) / self.head_dim)
        index = torch.arange(self.max_len).unsqueeze(1)
        theta_pos = index * theta_i
        self.cos_cache = torch.cos(theta_pos) # max_len * d/2
        self.sin_cache = torch.sin(theta_pos)

    def forward(self, X):
        # shape of X: batch_size, num_head, seq_len, head_dim
        _, _, seq_len, _ = X.shape
        cos_emb = torch.repeat_interleave(self.cos_cache[:seq_len], repeats=2, dim=1)
        sin_emb = torch.repeat_interleave(self.sin_cache[:seq_len], repeats=2, dim=1)

        sin_x = torch.stack([-X[..., 1::2], X[..., ::2]], dim=-1)
        
        return cos_emb * X + sin_emb * sin_x

In [ ]:
class MultiHeadLatentAttention(nn.Module):
    def __init__(self, hidden_size, num_heads, down_dim=64, up_dim=128, rope_head_dim=26):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.kv_head_dim = up_dim // num_heads
        self.rope_head_dim = rope_head_dim

        # weights needed
        self.down_proj_q = nn.Linear(hidden_size, down_dim)
        self.down_proj_kv = nn.Linear(hidden_size, down_dim)
        self.up_proj_q = nn.Linear(down_dim, up_dim)
        self.up_proj_k = nn.Linear(down_dim, up_dim)
        self.up_proj_v = nn.Linear(down_dim, up_dim)
        self.proj_qr = nn.Linear(down_dim, rope_head_dim*num_heads)
        self.proj_kr = nn.Linear(hidden_size, rope_head_dim)

        # Rope layer
        self.rope_k = RotaryEmbedding(rope_head_dim*num_heads, num_heads)
        self.rope_q = RotaryEmbedding(rope_head_dim, 1)

        self.out_proj = nn.Linear(self.kv_head_dim * num_heads, hidden_size)

    def forward(self, X: torch.tensor, mask=None):
        bs, seq_len, _ = X.shape

        kv_c = self.down_proj_kv(X)
        q_c = self.down_proj_q(X)
        k_up = self.up_proj_k(kv_c).view(bs, seq_len, self.num_heads, self.kv_head_dim).transpose(1,2)
        v_up = self.up_proj_v(kv_c).view(bs, seq_len, self.num_heads, self.kv_head_dim).transpose(1,2)
        q_up = self.up_proj_q(q_c).view(bs, seq_len, self.num_heads, self.kv_head_dim).transpose(1,2)

        q_r = self.proj_qr(q_c).view(bs, seq_len, self.num_heads, self.rope_head_dim).transpose(1,2)
        q_r = self.rope_q(q_r)
        k_r = self.proj_kr(X).unsqueeze(1) # bs, 1, seq_len, rope_head_dim
        k_r = self.rope_k(k_r).expand(bs, self.num_heads, seq_len, self.rope_head_dim)

        q = torch.cat([q_up, q_r], dim=-1)
        k = torch.cat([k_up, k_r], dim=-1)
        p = torch.matmul(q, k.transpose(-1,-2))
        p = p / (self.head_dim+self.rope_head_dim) ** 0.5

        if mask is not None:
            p = p.masked_fill(mask, self.float('-inf'))
        
        o = torch.matmul(torch.softmax(p, dim=-1), v_up) # bs, num_head, seq_len, self.kv_head_dim

        # concat
        o = o.transpose(1,2).reshape(bs, seq_len, -1)
        o = self.out_proj(o)
        return o

## Summary

Definitions:
- $d$: dimentions per head
    - $d_c$: KV compression dimension in MLA
    - $d_r$: rope dimension in MLA
- $n$
    - $n_h$: number of head
    - $n_g$: number of attention group
- $l$: number of layers in one model

| Method |  KV Cache (Per token)  |
| :--:   | :--:              |
| MHA    |   $2n_hdl$   |
|GQA     |   $2n_gdl$   |
|MQA     |  $2dl$       |
|MLA     |  $(d_c+d_r)l$| 

## References

1. (paper) Attention is all you need, https://arxiv.org/pdf/1706.03762
2. (paper) DeepSeek-V2: A Strong, Economical, and Efficient Mixture-of-Experts Language Model, https://arxiv.org/pdf/2405.04434